In [0]:
import os
import sys

import mlflow
from pyspark import cloudpickle
from threadpoolctl import threadpool_limits

sys.path.insert(0, os.path.abspath(".."))
from modelos import pronostico as P
from modelos.pronostico import backtest, para_spark, preparar

# Spark manda la función de applyInPandas a los ejecutores serializada con cloudpickle. Así viaja con el código
# del módulo, y los ejecutores no necesitan encontrar src/modelos por su cuenta. La función usa backtest,
# para_spark y preparar, y no P: el módulo viajaría entero, con su cargador de archivos del workspace, que es de
# dbruntime, y los ejecutores fallarían con "No module named 'dbruntime'
cloudpickle.register_pickle_by_value(P)

DESDE, CADA = "2022-01-02", 4
VARIABLES = [v for v in P.VARIABLES if v not in ("provincia_codigo", "poblacion")]   # con una provincia no dicen nada
usuario = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/vigia-dengue-modelo")

In [0]:
def backtest_de_una_provincia(features):
    """Recibe las features de una provincia como DataFrame de pandas y devuelve su backtest con un modelo propio."""
    with threadpool_limits(limits=1):   # un hilo por tarea: las provincias ya corren en paralelo
        f = preparar(features)
        return para_spark(backtest(f, DESDE, variables=VARIABLES, cada=CADA, modelo="gbm_provincia"))


esquema = "modelo string, dpa_provin string, emision date, horizonte bigint, pronostico double, origen date"
por_provincia = (spark.table("vigia.gold.features_provincia_semana")
                 .repartition(24, "dpa_provin")   # sin esto, Spark junta estas pocas filas en una sola tarea
                 .groupBy("dpa_provin")
                 .applyInPandas(backtest_de_una_provincia, schema=esquema)
                 .toPandas())
print(f"{len(por_provincia)} pronósticos de {por_provincia['dpa_provin'].nunique()} provincias")

In [0]:
features = spark.table("vigia.gold.features_provincia_semana").toPandas()
canal = spark.table("vigia.gold.canal_endemico").toPandas()
semanas = (spark.table("vigia.gold.dim_calendario_epi")
           .select("anio_epi", "semana_epi", "inicio_semana").distinct().toPandas())
base = P.evaluacion(P.preparar(features), canal, semanas, DESDE)
anteriores = P.de_spark(spark.table("vigia.ml.backtest_pronosticos")
                        .where("modelo IN ('canal', 'gbm_global')")
                        .select("modelo", "dpa_provin", "emision", "horizonte", "pronostico", "origen").toPandas())
pronosticos = P.unir(base, [anteriores, P.de_spark(por_provincia)])
error = P.error_por_horizonte(pronosticos).set_index("modelo")

with mlflow.start_run(run_name="backtest_gbm_provincia"):
    mlflow.log_params({"modelo": "gbm_provincia", "desde": DESDE, "reentrenar_cada": CADA,
                       "semanas_etiqueta": P.SEMANAS_ETIQUETA, "variables": len(VARIABLES), **P.HIPERPARAMETROS})
    mlflow.log_metrics(error.loc["gbm_provincia", ["mae_h1", "mae_h2", "mae_h3", "mae_h4", "mae_medio","mae_relativo"]].to_dict())

(spark.createDataFrame(P.para_spark(pronosticos[pronosticos["modelo"] == "gbm_provincia"]))
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("vigia.ml.backtest_por_provincia"))
display(error.round(2).reset_index())

In [0]:
a_2_4 = pronosticos[pronosticos["evaluable"] & pronosticos["horizonte"].between(2, 4)]
comparacion = (a_2_4.assign(error=(a_2_4["pronostico"] - a_2_4["real"]).abs())
               .pivot_table(index=["dpa_provin", "provincia"], columns="modelo", values="error", aggfunc="mean"))
comparacion["gana"] = (comparacion["gbm_provincia"] < comparacion["gbm_global"]).map({True: "provincia",                                                                       False: "global"})
display(comparacion.round(2).reset_index())